# 04 — 3-Stage Geometric Pruning & Non-Degenerate Rigid Triplet Extraction

This notebook demonstrates **Stage 4** of the `monocular-3d-gait-tracking-deltav2` pipeline (`src/postprocessing/geometric_pruner.py`):
1. **Stage 1 (Spatial DBSCAN):** Anisotropic depth scaling ($z_{\text{weight}} = 0.25$) to attenuate monocular Z-spikes.
2. **Stage 2 (Kinematic MAD Consensus):** Velocity and acceleration MAD filtering with a $0.015\text{ m/frame}$ physical noise floor.
3. **Stage 3 (Rigid Triplet Selection):** Enforcing minimum altitude $h_{\min} \ge 3.5\text{ cm}$ and aspect ratio $\text{AR} \le 8.0$ against normal-vector twist.

In [ ]:
import sys
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.postprocessing.geometric_pruner import GeometricPruner
from src.metrics.geometry.rigidity import RigidityMetric
from src.metrics.geometry.smoothness import SmoothnessMetric

In [ ]:
# Compare Raw vs Pruned Cloud vs Refined Triplet on P27_E2T2_90.mp4
raw_npz = PROJECT_ROOT / 'data/inference_results/raw/delta_v2_raw_P27_E2T2_90.mp4.npz'
ref_npz = PROJECT_ROOT / 'data/inference_results/refined/delta_v2_refined_P27_E2T2_90.mp4.npz'
if raw_npz.exists() and ref_npz.exists():
    raw_d = np.load(raw_npz, allow_pickle=True)
    ref_d = np.load(ref_npz, allow_pickle=True)
    raw_thigh = raw_d['coords'][:, :int(raw_d['n_thigh']), :]
    pruned_cloud = ref_d['full_cloud_coords']
    triplet = ref_d['coords']
    rig = RigidityMetric()
    print(f'Raw Thigh Cloud Shape    : {raw_thigh.shape} | Deformation: {rig.calculate(raw_thigh):.1f}%')
    print(f'Pruned Thigh Cloud Shape : {pruned_cloud.shape} | Deformation: {rig.calculate(pruned_cloud):.1f}%')
    print(f'Refined Rigid Triplet    : {triplet.shape} | Deformation: {rig.calculate(triplet):.1f}%')